# Build SMILES-based model to train for molecular generation.

This notebook shows the training of a SMILES-based generative model to output valid, drug-like molecules.

In [1]:
import pandas as pd
from rdkit import Chem
from rdkit import RDLogger

# Get the main logger
rdkit_logger = RDLogger.logger()

# Set its level to ERROR so only serious messages show up
rdkit_logger.setLevel(RDLogger.ERROR)
from tqdm import tqdm
import re

import pathlib
from pathlib import Path

import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

#verify Cuda is actually working

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version (runtime):", torch.version.cuda)

print("Torch CUDA libraries loaded:")
for lib in torch.cuda._CudaBase.__module__.split(":"):
    print(lib)

Torch version: 2.8.0+cu128
CUDA available: True
CUDA version (runtime): 12.8
Torch CUDA libraries loaded:
torch.cuda


## Setup.

In [19]:
readout = 'chembl_molecules'

In [20]:
# define paths
HERE = Path(pathlib.Path.cwd())
DATA = HERE / f"data_{readout}"
DATA.mkdir(parents=True, exist_ok=True)

In [21]:
df = pd.read_csv(f'{DATA}/class_a_gpcr_bioactivity.csv')

In [23]:
# Optional: Filter invalid SMILES
def is_valid_smiles(smi):
    try:
        return Chem.MolFromSmiles(smi) is not None
    except:
        return False

df = df[df['smiles'].apply(is_valid_smiles)].reset_index(drop=True)
print(f"Loaded {len(df)} valid SMILES.")

Loaded 99645 valid SMILES.


In [24]:
from rdkit.Chem.MolStandardize import rdMolStandardize

# Initialize standardizer components
fragment_remover = rdMolStandardize.FragmentRemover()
uncharger = rdMolStandardize.Uncharger()

def standardize_smiles(smi):
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        return None
    # Remove salts
    mol = fragment_remover.remove(mol)
    # Uncharge
    mol = uncharger.uncharge(mol)
    return Chem.MolToSmiles(mol)

In [25]:
df['standardised_smiles'] = df['smiles'].apply(standardize_smiles)
df = df[df['smiles'].notnull()].reset_index(drop=True)
print(f"After standardization: {len(df)} SMILES")

After standardization: 99645 SMILES


In [26]:
# Simple regex-based tokenizer for SMILES
def tokenize_smiles(smiles):
    # Basic regex for tokenizing SMILES
    pattern =  "(\[[^\[\]]{1,6}\])" + \
               "|Br|Cl" + \
               "|Si|Se|Na|Li|Ca|Fe|Zn|Cu" + \
               "|[B-Zb-z]" + \
               "|\d+" + \
               "|=|#|\(|\)|\.|:|\/|\\|\+|\-|\%|\@|\?"  # extended syntax
    regex = re.compile(pattern)
    tokens = regex.findall(smiles)
    return tokens

# Build vocabulary
from collections import Counter

token_counts = Counter()
for smi in tqdm(df['smiles']):
    tokens = tokenize_smiles(smi)
    token_counts.update(tokens)

# Add special tokens
special_tokens = ['<pad>', '<bos>', '<eos>', '<unk>']
vocab = special_tokens + sorted(token_counts.keys())
token_to_idx = {token: idx for idx, token in enumerate(vocab)}
idx_to_token = {idx: token for token, idx in token_to_idx.items()}
vocab_size = len(vocab)

print(f"Vocab size: {vocab_size}")

100%|██████████| 99645/99645 [00:00<00:00, 187137.06it/s]

Vocab size: 44


## Data structures and loading.

In [28]:
import torch
from torch.utils.data import Dataset

class SmilesDataset(Dataset):
    def __init__(self, smiles_list, token_to_idx, max_len=128):
        self.smiles_list = smiles_list
        self.token_to_idx = token_to_idx
        self.max_len = max_len

    def __len__(self):
        return len(self.smiles_list)

    def __getitem__(self, idx):
        smiles = self.smiles_list[idx]
        tokens = tokenize_smiles(smiles)
        tokens = ['<bos>'] + tokens + ['<eos>']
        token_ids = [self.token_to_idx.get(tok, self.token_to_idx['<unk>']) for tok in tokens]

        if len(token_ids) > self.max_len:
            token_ids = token_ids[:self.max_len]
        else:
            token_ids += [self.token_to_idx['<pad>']] * (self.max_len - len(token_ids))

        input_ids = torch.tensor(token_ids[:-1], dtype=torch.long)
        target_ids = torch.tensor(token_ids[1:], dtype=torch.long)

        return input_ids, target_ids


In [ ]:
from torch.utils.data import DataLoader

from sklearn.model_selection import train_test_split

train_smiles, val_smiles = train_test_split(df['smiles'].tolist(), test_size=0.1, random_state=42)

train_dataset = SmilesDataset(train_smiles, token_to_idx, max_len=128)
val_dataset = SmilesDataset(val_smiles, token_to_idx, max_len=128)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)


## Model architecture.

In [31]:
import torch.nn as nn

class SmilesRNN(nn.Module):
    def __init__(self, vocab_size, embed_dim=256, hidden_dim=512, num_layers=2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=token_to_idx['<pad>'])
        self.gru = nn.GRU(embed_dim, hidden_dim, num_layers=num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x, hidden=None):
        x = self.embedding(x)
        output, hidden = self.gru(x, hidden)
        logits = self.fc(output)
        return logits, hidden


In [ ]:
import torch

def sample_smiles(model, max_len=100, temperature=1.0):
    model.eval()
    input_seq = torch.tensor([[token_to_idx['<bos>']]], dtype=torch.long).to(device)
    hidden = None
    generated = []

    for _ in range(max_len):
        logits, hidden = model(input_seq, hidden)
        logits = logits[:, -1, :] / temperature  # use last token's logits
        probs = torch.softmax(logits, dim=-1)
        token_id = torch.multinomial(probs, num_samples=1).item()
        token = idx_to_token.get(token_id, '<unk>')

        if token == '<eos>' or token == '<pad>':
            break

        generated.append(token)
        input_seq = torch.tensor([[token_id]], dtype=torch.long).to(device)

    return ''.join(generated)


## Training loop

In [ ]:
import os
import torch.optim as optim
import torch.nn.functional as F

def evaluate(model, val_loader):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for input_ids, target_ids in val_loader:
            input_ids = input_ids.to(device)
            target_ids = target_ids.to(device)
            logits, _ = model(input_ids)
            loss = F.cross_entropy(logits.view(-1, vocab_size), target_ids.view(-1), ignore_index=token_to_idx['<pad>'])
            total_loss += loss.item()
    return total_loss / len(val_loader)

def generate_smiles_samples(model, n=5):
    samples = [sample_smiles(model) for _ in range(n)]
    return samples

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = SmilesRNN(vocab_size).to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3)

n_epochs = 10  # adjust as needed
save_dir = "smiles_rnn_model"
os.makedirs(save_dir, exist_ok=True)

for epoch in range(n_epochs):
    model.train()
    total_loss = 0

    for input_ids, target_ids in train_loader:
        input_ids = input_ids.to(device)
        target_ids = target_ids.to(device)

        optimizer.zero_grad()
        logits, _ = model(input_ids)
        loss = F.cross_entropy(logits.view(-1, vocab_size), target_ids.view(-1), ignore_index=token_to_idx['<pad>'])
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    train_loss = total_loss / len(train_loader)
    val_loss = evaluate(model, val_loader)
    print(f"Epoch {epoch+1} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

    # Generate example molecules
    samples = generate_smiles_samples(model, n=5)
    print("Samples:")
    for smi in samples:
        print(f"  {smi}")

    # Save checkpoint
    torch.save(model.state_dict(), os.path.join(save_dir, f"model_epoch_{epoch+1}.pt"))


# Eval